# ModernBERT smell-token classifier training (46 routed heads)

Notebook-first training workflow for the `role_section_excerpts*.jsonl` dataset.

In [ ]:
from pathlib import Path
import json
import math
import torch

from training_inference.data_utils import (
    create_dataset_build, create_tokenizer, PretokenizedSmellDataset, TokenClassificationCollator, summarize_dataset
)
from training_inference.model_pipeline import SharedEncoderSmellHeads, build_optimizer
from training_inference.train_loop import set_global_seed, make_dataloader, train_model

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'training_inference').exists():
    # If running from repo root's parent, adjust here manually.
    raise RuntimeError('Run this notebook from the repo root (the directory that contains training_inference/).')


In [ ]:
# ==== User-editable settings (simple variables, no heavy config system) ====
SEED = 42
BACKBONE_NAME = 'answerdotai/ModernBERT-base'  # or ModernBERT-large if you have enough VRAM
MAX_LENGTH = 8192

TRAIN_RATIO = 0.8
VAL_RATIO = 0.1
TEST_RATIO = 0.1
NEG_POS_RATIO = 2.0        # None disables downsampling. Example: 1.0 (1:1), 2.0 (2:1)
BALANCE_PER_SMELL = False  # True = apply ratio per smell bucket

BATCH_SIZE_TRAIN = 1       # long context model; start conservative
BATCH_SIZE_EVAL = 1
NUM_EPOCHS = 3
LR = 2e-5
WEIGHT_DECAY = 0.01
DROPOUT = 0.1
GRAD_ACCUM_STEPS = 8
MAX_GRAD_NORM = 1.0
NUM_WORKERS = 0
USE_AMP = True

# Optional: class weights for routed token CE loss (helps with O dominance)
USE_CLASS_WEIGHTS = False
O_WEIGHT = 0.25

OUTPUT_DIR = REPO_ROOT / 'training_inference' / 'runs' / 'modernbert_routed_46h'
RESUME_CHECKPOINT = None  # e.g., OUTPUT_DIR / 'checkpoints' / 'epoch_001.pt'


In [ ]:
# ==== Determinism ====
set_global_seed(SEED, deterministic=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device =', device)


In [ ]:
# ==== Load dataset / splits / mappings ====
dataset_build = create_dataset_build(
    repo_root=REPO_ROOT,
    preferred_dataset_name='role_section_excerpts.line.jsonl',
    train_ratio=TRAIN_RATIO,
    val_ratio=VAL_RATIO,
    test_ratio=TEST_RATIO,
    seed=SEED,
    train_neg_pos_ratio=NEG_POS_RATIO,
    balance_per_smell=BALANCE_PER_SMELL,
    split_manifest_dir=OUTPUT_DIR / 'split_manifests',
)

print('Train:', summarize_dataset(dataset_build.train_examples))
print('Val  :', summarize_dataset(dataset_build.val_examples))
print('Test :', summarize_dataset(dataset_build.test_examples))
print('num labels:', len(dataset_build.label_maps.label_to_id))
print('num smell heads:', len(dataset_build.smell_maps.smell_to_head))
assert len(dataset_build.smell_maps.smell_to_head) == 46, 'Expected 46 smells/heads.'

(OUTPUT_DIR / 'metadata').mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'metadata' / 'label_to_id.json').write_text(
    json.dumps(dataset_build.label_maps.label_to_id, ensure_ascii=False, indent=2), encoding='utf-8'
)
(OUTPUT_DIR / 'metadata' / 'smell_to_head.json').write_text(
    json.dumps(dataset_build.smell_maps.smell_to_head, ensure_ascii=False, indent=2), encoding='utf-8'
)


In [ ]:
# ==== Tokenizer + torch datasets ====
tokenizer = create_tokenizer(BACKBONE_NAME, use_fast=True, truncation_side='right')  # right truncation => truncate from end

train_ds = PretokenizedSmellDataset(
    dataset_build.train_examples, tokenizer, dataset_build.label_maps, dataset_build.smell_maps, max_length=MAX_LENGTH
)
val_ds = PretokenizedSmellDataset(
    dataset_build.val_examples, tokenizer, dataset_build.label_maps, dataset_build.smell_maps, max_length=MAX_LENGTH
)
test_ds = PretokenizedSmellDataset(
    dataset_build.test_examples, tokenizer, dataset_build.label_maps, dataset_build.smell_maps, max_length=MAX_LENGTH
)

collator = TokenClassificationCollator(tokenizer=tokenizer, label_pad_id=-100)
train_loader = make_dataloader(train_ds, collator, batch_size=BATCH_SIZE_TRAIN, shuffle=True, num_workers=NUM_WORKERS)
val_loader = make_dataloader(val_ds, collator, batch_size=BATCH_SIZE_EVAL, shuffle=False, num_workers=NUM_WORKERS)
test_loader = make_dataloader(test_ds, collator, batch_size=BATCH_SIZE_EVAL, shuffle=False, num_workers=NUM_WORKERS)


In [ ]:
# ==== Optional class weights (for O dominance) ====
class_weights = None
if USE_CLASS_WEIGHTS:
    label_to_id = dataset_build.label_maps.label_to_id
    class_weights = torch.ones(len(label_to_id), dtype=torch.float32)
    if 'O' in label_to_id:
        class_weights[label_to_id['O']] = float(O_WEIGHT)
    print('class_weights:', {k: float(class_weights[v]) for k,v in label_to_id.items()})
else:
    print('class_weights disabled')


In [ ]:
# ==== Build model (shared encoder + 46 smell-specific heads) ====
model = SharedEncoderSmellHeads.from_hf_pretrained(
    backbone_name=BACKBONE_NAME,
    num_smells=len(dataset_build.smell_maps.smell_to_head),
    num_labels=len(dataset_build.label_maps.label_to_id),
    dropout=DROPOUT,
    trust_remote_code=False,
)
optimizer = build_optimizer(model, lr=LR, weight_decay=WEIGHT_DECAY)

print('Model ready. hidden_size=', model.hidden_size)


In [ ]:
# ==== Train ====
artifacts = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    device=device,
    id_to_label=dataset_build.label_maps.id_to_label,
    label_to_id=dataset_build.label_maps.label_to_id,
    smell_to_head=dataset_build.smell_maps.smell_to_head,
    tokenizer_name_or_path=BACKBONE_NAME,
    output_dir=OUTPUT_DIR,
    num_epochs=NUM_EPOCHS,
    grad_accum_steps=GRAD_ACCUM_STEPS,
    max_grad_norm=MAX_GRAD_NORM,
    amp_enabled=USE_AMP,
    class_weights=class_weights,
    resume_checkpoint_path=RESUME_CHECKPOINT,
    debug_preview_count=3,
)

print('best checkpoint:', artifacts.best_checkpoint_path)
print('last checkpoint:', artifacts.last_checkpoint_path)


In [ ]:
# ==== (Optional) quick test-set evaluation using best checkpoint (manual next step) ====
# You can load the best checkpoint in the inference notebook, or add a test evaluation cell here.
